# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}, License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all RecordSets and their @id fields
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"@id: {rs.id}\n  Name: {rs.name}\n  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, Name: {field.name}, DataType: {field.data_type}")
    print('-' * 50)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Record set and field `@id`s are used for reference.

In [ ]:
# Get all recordset @id fields
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
# Load each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Preview columns and first rows for the first record set
first_rs_id = record_set_ids[0] if record_set_ids else None
if first_rs_id and not dataframes[first_rs_id].empty:
    print(f"Sample columns in record set {first_rs_id}:\n", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates filtering, normalization, and grouping by field values.

In [ ]:
# --- EDA Example ---
# Choose a record set and relevant numeric field by their @id

# We'll use the first record set as an example
rs = record_sets[0] if record_sets else None
if rs is not None:
    df = dataframes[rs.id]
    # Find numeric fields (float/int)
    numeric_field = None
    for field in rs.fields:
        if field.data_type in ['schema:Float', 'schema:Integer', 'schema:Number']:
            if field.id in df.columns:
                numeric_field = field.id
                break
    if numeric_field is not None:
        print(f"Using numeric field: {numeric_field}")
        # Remove nulls
        filtered_df = df[df[numeric_field].notnull()].copy()
        # Example: filter on values > threshold (pick e.g. 10 for demonstration purposes)
        threshold = 10
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
        print(f"Filtered records in '{rs.id}' with {numeric_field} > {threshold}: {len(filtered_df)} rows")
        # Normalize
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # If there's a categorical or string field, demonstrate grouping
        group_field = None
        for field in rs.fields:
            if field.data_type == 'schema:Text' and field.id in df.columns and filtered_df[field.id].nunique() > 1:
                group_field = field.id
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field found in the first record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if rs is not None and numeric_field is not None and not filtered_df.empty:
    # Plot the distribution of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {rs.id}")
    plt.xlabel(numeric_field)
    plt.show()

    # If grouping done above, show bar plot
    if group_field and not grouped_df.empty:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.xticks(rotation=60)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we've loaded the FAIR^2 mass spectrometry dataset using the Croissant schema and explored its record sets using their `@id` fields.
* We demonstrated extraction and normalization of data for a numeric field and grouped data by relevant textual attributes.
* Further analysis and modeling could use this structure by referencing additional fields and record sets as needed.